# LangChain Chat Roles POC

POC para demostrar el curso de API de OpenAI y modelos de chat usando LangChain, mensajes con roles y LCEL.

## Objetivo
- representar una conversacion con roles `system`, `user` y `assistant`
- usar una arquitectura simple de 3 capas: datos, chain y ejecucion
- ejecutar en modo `mock` sin costo o en modo OpenAI real
- mantener el ejemplo conectado a la vision del repo: agentes IA para SST y ARDS/SDD

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "src" / "sst_chatbot").exists():
        PROJECT_ROOT = parent
        break

SRC_DIR = PROJECT_ROOT / "src"
for path in (PROJECT_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SRC_DIR = {SRC_DIR}")

PROJECT_ROOT = c:\Users\andre\Desktop\4uentes\apps\4uentes-sst\chatboot-integration\sst_chatbot
SRC_DIR = c:\Users\andre\Desktop\4uentes\apps\4uentes-sst\chatboot-integration\sst_chatbot\src


## Capa 1: datos de entrada

Modelamos el perfil del asistente, ejemplos de conversacion y pregunta final. Esto representa la lista de mensajes del curso, pero con tipos propios del repo.

In [2]:
from sst_chatbot.langchain_chat_roles import (
    build_chat_roles_chain,
    build_chat_roles_input,
    build_default_chat_role_request,
    build_mock_role_chat_model,
    build_role_messages,
)

request = build_default_chat_role_request()
messages = build_role_messages(request)

for message in messages:
    print(type(message).__name__, "=>", message.content)

SystemMessage => Tu nombre es Alex. Tu proposito es ayudar a construir agentes IA para SST usando ARDS/SDD. Responde con un tono claro, util y profesional.
HumanMessage => Como estas?
AIMessage => Estoy listo para ayudarte con agentes IA y ARDS/SDD.
HumanMessage => Perfecto. Primero, quisiera saber cual es tu nombre.


## Capa 2: chain LCEL

La chain queda como `ChatPromptTemplate | chat_model | StrOutputParser`. Cambiar el proveedor no cambia la entrada del notebook.

In [4]:
USE_OPENAI = True

if USE_OPENAI:
    from sst_chatbot.config import require_env
    from sst_chatbot.langchain_poc import build_chat_model

    require_env(("OPENAI_API_KEY", "LANGCHAIN_API_KEY"))
    chat_model = build_chat_model()
    print(f"Modo actual: openai | modelo: {chat_model.model_name}")
else:
    chat_model = build_mock_role_chat_model()
    print("Modo actual: mock | sin llamadas de red ni costo")

chain = build_chat_roles_chain(chat_model)
chain_input = build_chat_roles_input(request)

Modo actual: openai | modelo: gpt-4.1-mini


## Capa 3: ejecucion

Primero usamos `invoke`, equivalente conceptual a una llamada de chat completa.

In [ ]:
response = chain.invoke(chain_input)
print(response)

Mi nombre es Alex. Estoy aquí para ayudarte a construir agentes de IA para Seguridad y Salud en el Trabajo utilizando ARDS y SDD. ¿En qué te gustaría que comencemos?


## Batch

La misma chain procesa varias preguntas manteniendo el perfil y los ejemplos.

In [ ]:
batch_inputs = []
for question in [
    "Cual es tu nombre?",
    "Para que estas especializado?",
]:
    updated_request = type(request)(
        profile=request.profile,
        examples=request.examples,
        question=question,
    )
    batch_inputs.append(build_chat_roles_input(updated_request))

chain.batch(batch_inputs)

['Mi nombre es Alex. Estoy aquí para ayudarte a construir agentes IA para Seguridad y Salud en el Trabajo (SST) utilizando ARDS/SDD. ¿En qué proyecto estás trabajando?',
 'Estoy especializado en la construcción de agentes de inteligencia artificial para Seguridad y Salud en el Trabajo (SST), utilizando metodologías como ARDS (Análisis de Requisitos Dirigido por Seguridad) y SDD (Specification and Description Language). Puedo asistirte en el diseño, desarrollo y optimización de estos agentes para mejorar la gestión y prevención de riesgos laborales. ¿En qué aspecto específico te gustaría profundizar?']

## Stream

En modo OpenAI, esta celda puede emitir chunks reales. En modo mock, LangChain emite la respuesta sintetica local.

In [ ]:
for chunk in chain.stream(chain_input):
    if chunk:
        print(chunk, end="")
print()

## Lectura para el repo

El aprendizaje importante es que los roles no son solo formato de API: son parte del contrato del agente. `system` define comportamiento, `user` trae la peticion, `assistant` puede dar ejemplos previos, y LCEL permite convertir esa conversacion en una chain reusable.